### 프롬프트 템플릿


In [1]:
!pip --version

pip 26.2.1 from D:\kingSJ\hanwha_0902\ex_0914\.venv\Lib\site-packages\pip (python 3.12)



In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import logging
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_teddynote.messages import stream_response

In [ ]:
# 로깅 전역 설정
# basicConfig는 '프로그램 전체'의 로깅 기본값을 정함
# 주의: 최초1회만 적용된다. 이미 설정된 뒤 다시 불러도 무시됨
# (주피터에서 값을 바꿔도 안먹히면 커널 재식작이 필요한 이유)
logging.basicConfig(
    filename="app.log", # 이 파일에 기록, 지정하면 콘솔에는 안 찍히고 파일로만 간다
                        # 기본 모드가 'a'(append)라 실행할수록 계속 누적됨
    level=logging.INFO, # INFO 이상만 기록 -> DEBUG는 무시, INFO/WARNING/ERROR/CRITICAL 은 기록
    format="%(asctime)s - %(levelname)s - %(message)s", # 시작-레벨-메시지 형태로 한 줄씩 나음
    encoding="utf-8" # 한글이 깨지지 않게, 윈도우에서 필수
)

# 이름 붙은 로거를 가져온다. 로그 출처를 구분하기 위한 이름표
# foramt에 %(name)s 를 넣으면 이 이름이 함께 찍힌다.
logger = logging.getLogger("MyMultimodalApp")

# 시작 시점 기록, 나중에 로그를 볼 때 '이 지점부터 이번 실행'이라는 구분선이 된다
logger.info("LangSmith 연동 및 애플리케이션 시작2")

# try 블록 : 실패할 수 있는 작업들
# 네트워크 호출, 파일 접근처럼 터질 수 있는 코드를 감싼다
try:
    multimodal_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    IMAGE_URL = "https://t3.ftcdn.net/jpg/03/77/33/96/360_F_377339633_Rtv9I77sSmSNcev8bEcnVxTHrXB4nRJ5.jpg"
    prompt = "이미지의 내용을 한국어로 상세히 설명해 주세요"

    # 요청 '직전'에 입력값을 남기는 게 핵심
    # 여기서 터지면 로그에 남은 이 줄이 마지막 단서가 된다.
    logger.info(f"요청 전송 - prompt: {prompt} / Image: {IMAGE_URL}")

    # 멀티모달 메시지 : content가 문자열이 아니라 블록 리스트
    # 텍스트 블록 + 이미지 블록을 한 메시지에 담아 보냄
    message = HumanMessage(
        content=[
            {"type": "text", "text" : prompt},
            {"type" : "image_url", "image_url": {"url": IMAGE_URL}}
        ]
    )

    # stream()은 제너레이터를 '만들기만' 한다. 이 줄에서는 아직 호출이 안 일어남
    # 실제 통신은 아래 stream_response() 가 소비하기 시작할 때 발생
    answer = multimodal_llm.stream([message])

    logger.info("답변 수신 및 스트리밍 시작")
    stream_response(answer) # <- 여기서 실제로 토큰이 흘러나오며 출력됨
    logger.info("답변 처리 완료") # 이 줄이 남았다면 끝까지 성공

except Exception as e:
    # error 레벨로 기록 -> 나중에 로그에서 ERROR만 검색하면 사고 지점만 뽑아볼 수 있다.
    # exc_info=True가 핵심 : 예외 메시지뿐 아니라 '전체 스택 트레이스'까지 파일에 남긴다.
                            # 이게 없으면 어느 줄에서 터졌는지 알 수 없다.
    logger.error(f"실행 중 에러 발생: {str(e)}", exc_info=True)

이 이미지는 표 형식으로 구성된 데이터입니다. 표의 제목은 "TABLE 001: LOREM IPSUM DOLOR AMIS ENIMA ACCUMMER TUNA"입니다. 표는 여러 열로 나뉘어 있으며, 각 열은 다음과 같은 항목을 포함하고 있습니다:

1. **첫 번째 열**: 항목의 이름 또는 설명이 들어 있습니다.
2. **두 번째 열**: "Loremis"라는 제목 아래 숫자 데이터가 나열되어 있습니다.
3. **세 번째 열**: "Amis terim"이라는 제목 아래 비율 또는 퍼센트가 포함되어 있습니다.
4. **네 번째 열**: "Gato lepis"라는 제목 아래 'YES' 또는 'NO'와 같은 값이 있습니다.
5. **다섯 번째 열**: "Tortores"라는 제목 아래 금액이 표시되어 있습니다.

각 행은 특정 항목에 대한 정보를 제공하며, 숫자와 텍스트가 혼합되어 있습니다. 예를 들어, 첫 번째 행은 "Lorem dolor siamet"이라는 항목에 대해 8,288이라는 숫자와 123%라는 비율, 'YES'라는 값, 그리고 $89라는 금액을 보여줍니다.

표의 하단에는 "Lorem ipsum dolor sit amet..."이라는 라틴어 문장이 포함되어 있어, 일반적으로 텍스트의 예시로 사용되는 내용입니다. 이 문장은 표의 내용과는 직접적인 관련이 없으며, 디자인이나 레이아웃을 위한 더미 텍스트로 보입니다.